# Lab 1: Pré-requisitos e Configuração da Infraestrutura

## Visão Geral
Verifique todos os pré-requisitos para o workshop e implante a stack da aplicação CRM na AWS (EC2 + NGINX + DynamoDB).

## Objetivos
**Parte 1: Pré-requisitos**
- Verificar a versão do Python (3.10+)
- Verificar conta e credenciais AWS
- Instalar dependências do workshop
- Verificar o SDK do Bedrock AgentCore e o starter toolkit
- Testar acesso ao modelo Bedrock
- Configurar Agent Memory para contexto compartilhado
- Configurar identidades de Usuário e Agente usando Amazon Cognito

**Parte 2: Configuração da Infraestrutura**
- Provisionar infraestrutura AWS: instância EC2, NGINX, DynamoDB, CloudWatch
- Implantar uma aplicação CRM de exemplo
- Criar scripts de injeção de falhas para simular problemas
- Configurar monitoramento com CloudWatch
- Verificar se a infraestrutura está em execução e acessível

## O Que Você Aprenderá
- Pré-requisitos e fluxo de configuração do workshop
- Padrões de injeção de falhas para testar resposta a incidentes
- Configuração de logs e métricas do CloudWatch para diagnósticos

## 1. Verificar Versão do Python

In [ ]:
import sys
print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"
print("✅ Python version check passed")

## 2. Instalar Dependências do Workshop

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

## 3. Verificar Configuração da AWS

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, AWS_PROFILE, MODEL_ID, WORKSHOP_NAME
from lab_helpers.lab_01.infrastructure import get_app_url

# Display configuration
print(f"Workshop Name: {WORKSHOP_NAME}")
print(f"AWS Region: {AWS_REGION}")
print(f"Model ID: {MODEL_ID}\n")

# Verify AWS credentials
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sts = session.client('sts')
identity = sts.get_caller_identity()

print(f"✅ AWS Account: {identity['Account']}")
print(f"✅ AWS User/Role: {identity['Arn']}")

## 4. Testar Acesso ao Modelo Bedrock

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, MODEL_ID, AWS_PROFILE

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client('bedrock', region_name=AWS_REGION)

# Verify model access
try:
    model = bedrock.get_foundation_model(modelIdentifier=MODEL_ID)
    print(f"Model ID: {MODEL_ID}")
    print(f"✅ Bedrock model access verified")
except Exception as e:
    print(f"❌ Error accessing model: {e}")
    raise

## 5. Verificar Componentes do AgentCore

In [ ]:
import importlib

packages = ['bedrock_agentcore', 'strands', 'boto3', 'pydantic']

for package in packages:
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'installed')
        print(f"✅ {package:<20} {version}")
    except ImportError:
        print(f"❌ {package:<20} NOT FOUND")

print("\n✅ All core packages verified")

## Resumo
✅ Todos os pré-requisitos verificados. Pronto para prosseguir para o Lab 1: Configuração da Infraestrutura e Injeção de Falhas.

## Parte 1.5: Configuração do Cognito (Autenticação para Labs 3-5)

### Visão Geral

Nesta seção, configuraremos o AWS Cognito para a infraestrutura de autenticação utilizada nos Labs 3-5:

**O Que Criaremos:**
- Cognito User Pool: `aiml301-UserPool`
- **Dois Grupos de Usuários** (NOVO):
  - **developers**: Usuários que criam planos de remediação
  - **approvers**: Usuários que aprovam e executam planos
- **Dois App Clients**:
  - **User Auth Client** (público): Para autenticação de usuários finais com suporte a OAuth
  - **M2M Client** (confidencial): Para autenticação serviço-a-serviço entre Gateway e Runtime
- **Resource Server**: Escopos customizados para autorização refinada (`mcp.invoke`, `runtime.access`)
- **User Pool Domain**: Endpoint de token OAuth2
- **Dois Usuários de Teste**:
  - **Usuário Developer**: `testuser@aiml301.example.com` (membro do grupo `developers`)
  - **Usuário Approver**: `approver@aiml301.example.com` (membro do grupo `approvers`)

**Fluxos de Autenticação:**
1. **User Auth** (Client → Gateway): Usuários finais autenticam com credenciais, recebem tokens JWT com associação a grupos
2. **M2M Auth** (Gateway → Runtime): Gateway usa client credentials grant para obter tokens M2M para acesso ao Runtime

**Fluxo Multi-Ator (Lab-03):**
- **Developer** faz login → Cria plano de remediação → É bloqueado (precisa de aprovação)
- **Approver** faz login → Descobre incidentes pendentes → Revisa plano → Aprova execução
- **Developer** retorna → Vê aprovação na memória compartilhada → Executa etapas aprovadas

**Claims do Token JWT:**
Após a configuração, os tokens JWT ID incluirão:
```json
{
  "email": "developer@aiml301.example.com",
  "cognito:username": "developer@aiml301.example.com",
  "cognito:groups": ["developers"],
  "sub": "uuid",
  "scope": "openid profile email custom-scopes"
}
```

**Por Que Grupos?**
- **Autorização baseada em papéis**: Gateway pode verificar se o usuário está no grupo `approvers` antes de permitir execução
- **Roteamento de incidentes**: Notificar apenas approvers para incidentes pendentes
- **Trilhas de auditoria**: Registros de memória mostram qual papel executou cada ação
- **Identificação de ator**: A claim `email` fornece um actor_id legível em vez de UUID

### Objetivos

✅ Configurar infraestrutura Cognito para autenticação centralizada
✅ Criar grupos de usuários para controle de acesso baseado em papéis
✅ Habilitar modos de autenticação duplos: baseada em usuário e serviço-a-serviço
✅ Criar escopos de autorização refinados
✅ Habilitar fluxos OAuth para tokens JWT ID ricos
✅ Armazenar configuração no SSM Parameter Store para uso nos labs posteriores

#### 1. Executar Configuração do Cognito

In [ ]:
from lab_helpers.cognito_setup import setup_cognito_complete

# Execute complete Cognito setup workflow
cognito_config = setup_cognito_complete()

print("\n" + "="*70)
print("COGNITO SETUP COMPLETE")
print("="*70)
print("Cognito User Pool ID: ", cognito_config['user_pool_id'])

## Parte 1.6: Configuração de Memória para Labs 2-5

Nesta seção, criaremos um recurso compartilhado de AgentCore Memory que será usado por todos os labs de agentes (2-5) para histórico de conversas e gerenciamento de sessões.

### O Que Criaremos:
- Recurso AgentCore Memory com expiração de 7 dias
- Armazenar memory_id no Parameter Store para fácil acesso nos Labs 2-5
- Armazenar ID de sessão padrão para rastreamento estático de sessão nos Labs 2-4

### Aprendizado-Chave:
Memory permite persistência de contexto entre chamadas de agentes e conversas multi-turno. Todos os labs compartilharão este único recurso de memória.

### Objetivos
✅ Criar recurso AgentCore Memory  
✅ Armazenar configuração de memória no Parameter Store  
✅ Habilitar carregamento de histórico de conversas para agentes downstream

In [ ]:
### 1.6.1: Create AgentCore Memory Resource

from bedrock_agentcore.memory import MemoryClient
from lab_helpers.constants import PARAMETER_PATHS
from datetime import datetime

memory_client = MemoryClient(region_name=AWS_REGION)
memory_name = f"{PARAMETER_PATHS['memory']['memory_name_prefix']}_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Creating memory: {memory_name}")
memory = memory_client.create_memory_and_wait(
    name=memory_name,
    description="SRE Agent Shared Short-Term Memory for Labs 2-5",
    strategies=[],
    event_expiry_days=7,
    max_wait=600,
    poll_interval=10
)

memory_id = memory['id']
print(f"✅ Memory created: {memory_id} (Status: ACTIVE, Expiry: 7 days)")

In [ ]:
### 1.6.2: Store Memory Configuration in Parameter Store

from lab_helpers.parameter_store import put_parameter

# Store memory_id for Labs 2-5
put_parameter(
    PARAMETER_PATHS['memory']['memory_id'],
    memory_id,
    description="Memory ID for agent conversation history",
    region_name=AWS_REGION
)

# Store default session ID for Labs 2-4
put_parameter(
    PARAMETER_PATHS['memory']['default_session_id'],
    "crm-session-id",
    description="Default session ID for Labs 2-4",
    region_name=AWS_REGION
)

print(f"✅ PSM Keys stored:")
print(f"   • {PARAMETER_PATHS['memory']['memory_id']} = {memory_id}")
print(f"   • {PARAMETER_PATHS['memory']['default_session_id']} = crm-session-id")

### Resumo: Configuração de Memória Concluída

✅ **Recurso AgentCore Memory Criado**
- Recurso de memória compartilhado único para todos os labs (2-5)
- Expiração automática de 7 dias para gerenciamento de custos
- Suporta conversas multi-turno e carregamento de contexto

✅ **Configuração do Parameter Store**
- Memory ID armazenado para recuperação nos Labs 2-5
- ID de sessão padrão disponível para Labs 2-4
- Segue padrão de configuração centralizada

**Próximos Passos:**
- Lab 2: Recuperar memory_id e inicializar hooks de memória
- Lab 3-4: Usar a mesma memória para agentes de remediação/prevenção
- Lab 5: Orquestração multi-agente com memória compartilhada

## Parte 2: Configuração da Infraestrutura e Implantação da Aplicação CRM

A Configuração da Infraestrutura e Implantação da Aplicação CRM é automatizada como parte da configuração do workshop.

Por favor, prossiga para a próxima seção.

In [ ]:
# Try url with both port 80 and 8080
print(f"Click here to access the CRM App UI: '{get_app_url()}'")


## 1. Configurar Utilitários de Injeção de Falhas

Nesta seção, prepararemos ferramentas para injetar falhas na infraestrutura e revisar falhas pré-configuradas já incorporadas na implantação. O workshop inclui **4 falhas no total** para treinamento abrangente de SRE:

- **Falha 1: DynamoDB Throttling** - Reduzir a capacidade da tabela para acionar ProvisionedThroughputExceededException
- **Falha 2: Problemas de Permissão IAM** - Restringir permissões da role EC2 para causar erros AccessDenied

Essas falhas serão usadas ao longo do workshop para testar as capacidades de diagnóstico do seu agente SRE em diferentes modos de falha e métodos de detecção.

In [ ]:
from lab_helpers.lab_01.fault_injection import (
    initialize_fault_injection,
    inject_dynamodb_throttling,
    inject_iam_permissions,
)

# Initialize AWS clients and retrieve infrastructure resource IDs from SSM
print("Initializing fault injection utilities...")
resources = initialize_fault_injection(AWS_REGION, AWS_PROFILE)

print(f"\nDiscovered Infrastructure Resources:")
print(f"  Nginx Instance: {resources.get('nginx_instance_id', 'Not found')}")
print(f"  App Instance: {resources.get('app_instance_id', 'Not found')}")
print(f"  CRM Activities Table: {resources.get('crm_activities_table_name', 'Not found')}")
print(f"  CRM Customers Table: {resources.get('crm_customers_table_name', 'Not found')}")
print(f"  CRM Deals Table: {resources.get('crm_deals_table_name', 'Not found')}")
print(f"  EC2 Role: {resources.get('ec2_role_name', 'Not found')}")
print(f"  Public ALB DNS: {resources.get('public_alb_dns', 'Not found')}")

print("\n✅ Fault injection utilities ready")

## 2. Verificar Infraestrutura

Antes de injetar falhas, vamos verificar se a stack do CloudFormation criou todos os recursos necessários e se estão saudáveis.

In [ ]:
from lab_helpers.lab_01.infrastructure import (
    verify_ec2_instances,
    verify_dynamodb_tables,
    verify_alb_health,
    verify_cloudwatch_logs
)

print("Verifying infrastructure components...\n")

# Verify EC2 instances are running
ec2_status = verify_ec2_instances(resources, AWS_REGION, AWS_PROFILE)

# Verify DynamoDB tables exist and are accessible
dynamodb_status = verify_dynamodb_tables(resources, AWS_REGION, AWS_PROFILE)

# Verify ALB targets are healthy
alb_status = verify_alb_health(resources, AWS_REGION, AWS_PROFILE)

# Verify CloudWatch log groups exist
logs_status = verify_cloudwatch_logs(AWS_REGION, AWS_PROFILE)

if all([ec2_status, dynamodb_status, alb_status, logs_status]):
    print("\n✅ All infrastructure components verified and healthy")
else:
    print("\n⚠️  Some infrastructure components failed verification")

## 3. Testar Injeção de Falhas e Revisar Falhas Pré-Configuradas

Nesta seção, injetaremos duas falhas na infraestrutura e revisaremos duas falhas adicionais que estão pré-configuradas na implantação. Juntas, essas **4 falhas** fornecerão cenários abrangentes de treinamento para seus agentes de diagnóstico.


### Falha 1: DynamoDB Throttling

**O que é:**
DynamoDB throttling ocorre quando sua aplicação excede a capacidade provisionada de leitura/escrita das suas tabelas. Este é um problema comum em produção que pode acontecer quando:
- Picos de tráfego excedem inesperadamente a capacidade provisionada
- Tabelas são configuradas incorretamente com unidades de capacidade insuficientes

**Como injetamos esta falha:**
A função auxiliar `inject_dynamodb_throttling()` simula isso através de:
- Conversão da tabela de métricas de `PAY_PER_REQUEST` (ilimitado) para modo de cobrança `PROVISIONED`
- Definição de limites de capacidade extremamente baixos: **1 Read Capacity Unit** e **1 Write Capacity Unit**
- Isso significa que a tabela só consegue processar ~1 operação de leitura e ~1 de escrita por segundo
- Qualquer carga normal da aplicação excederá imediatamente esses limites

**Impacto esperado:**
- Erros `ProvisionedThroughputExceededException` nos logs da aplicação
- Aumento de latência conforme requisições são limitadas e retentadas
- Métricas do CloudWatch mostrarão requisições throttled


In [ ]:
# Execute DynamoDB throttling fault injection
success = inject_dynamodb_throttling(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ DynamoDB throttling fault injected successfully")
    print("   → Table converted to PROVISIONED mode with 1 RCU/1 WCU")
    print("   → Normal application load will now trigger throttling")
else:
    print("❌ Failed to inject DynamoDB throttling fault")

### Carregar Tabelas da Aplicação 

Agora vamos fazer um teste de carga no nosso endpoint. Enviaremos 20 requisições concorrentes/segundo durante 30 segundos. Esta carga não é muito significativa, mas devido à configuração incorreta na capacidade provisionada da tabela - nossa aplicação deverá mostrar erros `ProvisionedThroughputExceededException` nos [logs da aplicação](https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsre-workshop$252Fcrm-application) e as métricas do CloudWatch mostrarão requisições throttled. Se você tentar acessar a aba Customers durante o teste de carga, terá problemas com o carregamento dos dados.

In [ ]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor

alb_dns = resources['public_alb_dns']
url = f"http://{alb_dns}:8080/api/customers"

def make_request(i):
    try:
        requests.get(url, timeout=5)
    except:
        pass

for second in range(1, 31):
    with ThreadPoolExecutor(max_workers=50) as executor:
        executor.map(make_request, range(50))

    if second % 10 == 0:
        print(f"Progress: {second}/30 seconds")

    time.sleep(1)

print("\n✓ Load test complete")

### Falha 2: Problemas de Permissão IAM

**O que é:**
Problemas de permissão IAM ocorrem quando aplicações não possuem as permissões necessárias para acessar recursos AWS. Este é um dos problemas mais comuns em produção, frequentemente causado por:
- Políticas de segurança excessivamente restritivas aplicadas sem testes
- Falhas na assunção de roles devido a modificações na trust policy
- Equipe de segurança aplicando políticas de deny genéricas

**Como injetamos esta falha:**
Nossa função auxiliar `inject_iam_permissions()` simula isso através de:
- Localização da IAM role da instância EC2 usada pelos servidores da aplicação
- Backup da política original de acesso ao DynamoDB
- Substituição por uma política explícita de **Deny** para operações-chave do DynamoDB
- Alvo: `PutItem`, `GetItem`, `Query`, `Scan`, `UpdateItem`, `DeleteItem`
- Como políticas Deny sobrepõem políticas Allow, isso bloqueia imediatamente o acesso ao banco de dados

**Impacto esperado:**
- Exceções `AccessDenied` nos logs da aplicação para qualquer operação de banco de dados
- Falha completa de funcionalidades que requerem acesso ao DynamoDB


In [ ]:
# Execute IAM permission fault injection
success = inject_iam_permissions(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ IAM permission fault injected successfully")
    print(f"   → EC2 role '{resources.get('ec2_role_name', 'Unknown')}' now has Deny policy")
    print("   → All DynamoDB operations will return AccessDenied")
else:
    print("❌ Failed to inject IAM permission fault")

Vamos testar qual resposta obtemos ao invocar nossa aplicação agora. Devemos ver erro 500 devido aos problemas no backend com a API não conseguindo recuperar dados do DynamoDB.
**Nota**: Pode levar um minuto para as permissões IAM se propagarem. Se você não estiver vendo erros 500, por favor aguarde e tente novamente.

In [ ]:
time.sleep(180)
alb_dns = resources['public_alb_dns']

url = f"http://{alb_dns}:8080/api/deals"

print(f"\nGenerating 5 requests to trigger IAM errors...")
print(f"Target: {url}\n")

for i in range(10):
    try:
        response = requests.get(url, timeout=5)
        print(f"Request {i+1} - Status: {response.status_code}")
    except Exception as e:
        print(f"Request {i+1} - Error: {str(e)}")

    time.sleep(1)  # Small delay to avoid overwhelming

print("\n✓ Load complete - waiting 10 seconds for logs to propagate...")
time.sleep(10)


## Resumo

✅ Pré-requisitos verificados e infraestrutura implantada. A aplicação CRM está em execução e monitorada via CloudWatch. Injetamos falhas simulando problemas reais de produção para o nosso Agente diagnosticar.

Próximo: Lab 2 - Construir o Agente de Diagnóstico (Lab-02-diagnostics-agent.ipynb)

In [ ]:
# Try url with both port 80 and 8080
print(f"Click here to access the CRM App UI: '{get_app_url()}'")
